# Session 2: a day of load, and a bill with slabs
**A. Real-world problem.** Two homes can use the same kWh and still stress the wires
differently: one runs everything at once, the other spreads the load. And a bill is rarely
"kWh x one rate": the price per unit steps up as you use more (slabs). Can you cut a bill by
20 percent, and how much usage must go?

*The tariff below is an ILLUSTRATIVE teaching tariff, not any real utility's. Educational
model, not an electrical design tool.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

HOURS = np.arange(24)
HEATER_KW = 1.5
SLABS = [(100.0, 3.0), (100.0, 5.0), (None, 7.5)]   # (slab size in kWh, Rs per kWh); None = the rest
FIXED = 50.0                                        # Rs per month, illustrative
DAYS = 30
VOLTS = 230.0

## Step 1: when each appliance is on
Each appliance has power (kW) and the hours it runs. The profile is the sum over appliances
of power in each hour. One hour at P kW is P kWh, so daily energy = sum of the 24 values.
The heater runs from `start` for `dur` hours; the overlap of the heater interval with each
hour tells us the fraction of that hour it is on.

In [ ]:
def overlap(hour, start, dur):
    """Hours of [hour, hour+1) that fall inside the interval [start, start+dur), wrapping at 24."""
    total = 0.0
    for shift in (0.0, -24.0):
        lo = max(hour, start + shift)
        hi = min(hour + 1, start + dur + shift)
        total += max(0.0, hi - lo)
    return total


def appliance_profiles(start=6.0, dur=2.0, scale=1.0, heater_kw=HEATER_KW):
    """kW in each hour for each appliance. `scale` multiplies fans, lights and TV."""
    h = HOURS
    return {
        "Fridge": np.full(24, 0.12),
        "Fans": np.where((h >= 11) & (h <= 18), 0.21, 0.0) * scale,
        "Lights": np.where((h >= 18) & (h <= 22), 0.06, 0.0) * scale,
        "TV": np.where((h >= 19) & (h <= 22), 0.10, 0.0) * scale,
        "Heater": np.array([heater_kw * overlap(x, start, dur) for x in h]),
    }


def load_profile(start=6.0, dur=2.0, scale=1.0, heater_kw=HEATER_KW):
    return sum(appliance_profiles(start, dur, scale, heater_kw).values())


def daily_kwh(profile):
    return float(np.sum(profile) * 1.0)   # 1 hour per entry

## Step 2: slab bill, peak, power factor
Bill = fixed charge + sum over slabs (kWh in the slab x its rate). Peak demand is the
largest hourly load. For alternating current, real power P (kW) relates to apparent power
S (kVA) by the power factor pf = P / S, and the current is I = P / (V x pf).

In [ ]:
def slab_bill(kwh, slabs=SLABS, fixed=FIXED, rate_mult=1.0):
    if kwh < 0:
        raise ValueError("energy cannot be negative")
    bill, left = fixed, kwh
    for size, rate in slabs:
        used = left if size is None else min(left, size)
        bill += used * rate * rate_mult
        left -= used
    return float(bill)


def peak_kw(profile):
    return float(np.max(profile))


def apparent_kva(kw, pf):
    return kw / pf


def current_amps(kw, pf, volts=VOLTS):
    return kw * 1000.0 / (volts * pf)


def monthly_bill(profile, days=DAYS):
    return slab_bill(daily_kwh(profile) * days)

## Step 3: the design problem, cut the bill by 20 percent
Find the heater time that gives 80 percent of today's bill. The bill rises with usage, so
bisection (halve the search interval until it fits) works.

In [ ]:
def heater_hours_for_bill(target_bill, start=6.0, scale=1.0, hi=4.0):
    f = lambda d: monthly_bill(load_profile(start, d, scale)) - target_bill
    lo = 0.0
    if f(lo) > 0:
        return None            # impossible even with the heater off
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if f(mid) < 0 else (lo, mid)
    return 0.5 * (lo + hi)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    prof = appliance_profiles()
    total = load_profile()
    month = daily_kwh(total) * DAYS
    base = monthly_bill(total)
    print(f"daily {daily_kwh(total):.2f} kWh, month {month:.1f} kWh, bill Rs {base:.1f}")
    pk = peak_kw(total)
    print(f"peak {pk:.2f} kW, at pf 0.9: {apparent_kva(pk, 0.9):.2f} kVA, {current_amps(pk, 0.9):.2f} A")
    d = heater_hours_for_bill(0.8 * base)
    new = load_profile(6.0, d)
    print(f"heater time for -20% bill: {d:.3f} h  (bill Rs {monthly_bill(new):.1f}, target {0.8 * base:.1f})")
    print(f"energy cut needed: {100 * (1 - daily_kwh(new) / daily_kwh(total)):.1f} %")

    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.stackplot(np.r_[HOURS, 24], *[np.r_[v, v[-1]] for v in prof.values()], labels=list(prof.keys()), step="post")
    ax.set_xlabel("hour of day (h)")
    ax.set_ylabel("power (kW)")
    ax.set_title("24-hour load profile, stacked by appliance")
    ax.set_xlim(0, 24)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "load_profile.png", dpi=120)

    kwh = np.linspace(0, 400, 401)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.plot(kwh, [slab_bill(k) for k in kwh])
    ax2.plot([month], [base], "o", color="red", label=f"this home: {month:.0f} kWh, Rs {base:.0f}")
    ax2.axhline(0.8 * base, color="gray", ls="--", label="20% cut target")
    ax2.set_xlabel("energy per month (kWh)")
    ax2.set_ylabel("monthly bill (Rs, illustrative tariff)")
    ax2.set_title("Slab tariff: the slope steps up")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "slab_bill.png", dpi=120)

## Coding tasks
1. **Easy.** Print the load profile for a heater at 19:00 with the same 2 h. Predict the peak hour and the peak kW, and check that the energy is unchanged (Section F, A).
2. **Stretch.** Change `SLABS` to a flat rate of 6 Rs per kWh and see how the cost of the 20 percent cut changes. Which tariff rewards saving the last units more?
3. **Explain.** Explain why the bill falls 20 percent while the energy falls only 13 percent, using the marginal rate.

## Assessment
- **A. Conceptual.** Why can two homes with the same monthly kWh have different peak demand? Why does the peak matter even though the bill is only for kWh?
- **B. Numerical.** A home uses 150 kWh in a month. Find its bill with the slab tariff and its average price per kWh. Check that the answer is sensible.
- **C. Graph and simulation interpretation.** On the bill graph, how much more does 10 extra kWh cost at 250 kWh than at 50 kWh? Why?
- **D. Design and optimisation challenge.** Design: cut the monthly bill from Rs 1208.50 by 20 percent. Find the heater time needed if nothing else changes, then find a second plan that also cuts the fans, lights and TV by 20 percent. State the energy cut each needs and the trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    prof = L.load_profile()
    checks.append(("profile has 24 hourly values", len(prof) == 24))
    kwh = L.daily_kwh(prof)
    checks.append((f"daily energy = 2.88 + 1.68 + 0.30 + 0.40 + 3.00 = 8.26 kWh (got {kwh:.2f})", abs(kwh - 8.26) < 1e-9))
    # hand calculation of the slab bill for 247.8 kWh
    hand = 50 + 100 * 3.0 + 100 * 5.0 + 47.8 * 7.5
    bill = L.monthly_bill(prof)
    checks.append((f"worked example: 247.8 kWh costs 50+300+500+358.5 = Rs 1208.50 (got {bill:.2f})", abs(bill - hand) < 1e-6))
    checks.append(("slab bill at 0 kWh is the fixed charge", abs(L.slab_bill(0.0) - 50.0) < 1e-12))
    checks.append(("slab bill at exactly 100 kWh is 50 + 300", abs(L.slab_bill(100.0) - 350.0) < 1e-12))
    checks.append(("bill never decreases as usage grows", all(L.slab_bill(a) <= L.slab_bill(a + 1) for a in range(0, 400))))
    try:
        L.slab_bill(-1.0)
        bad = False
    except ValueError:
        bad = True
    checks.append(("negative energy is rejected", bad))
    checks.append(("peak demand is 0.12 + 1.5 = 1.62 kW at 6-7 h", abs(L.peak_kw(prof) - 1.62) < 1e-9 and int(np.argmax(prof)) == 6))
    checks.append(("current at 1.62 kW, pf 0.9, 230 V is 7.83 A", abs(L.current_amps(1.62, 0.9) - 7.826) < 0.01))
    checks.append(("unity power factor: kVA equals kW", abs(L.apparent_kva(1.62, 1.0) - 1.62) < 1e-12))
    checks.append(("a heater from 23:00 for 2 h wraps past midnight", abs(L.overlap(0, 23.0, 2.0) - 1.0) < 1e-12 and abs(L.overlap(23, 23.0, 2.0) - 1.0) < 1e-12))
    checks.append(("moving the heater keeps the energy, changes the peak hour", abs(L.daily_kwh(L.load_profile(19.0)) - kwh) < 1e-9))
    d = L.heater_hours_for_bill(0.8 * bill)
    new = L.monthly_bill(L.load_profile(6.0, d))
    checks.append((f"design: heater {d:.3f} h gives 80% of the bill", abs(new - 0.8 * bill) < 1e-3))
    cut = 1 - L.daily_kwh(L.load_profile(6.0, d)) / kwh
    checks.append((f"a 20% bill cut needs only a {100 * cut:.1f}% energy cut (the top slab is dearer)", 0.12 < cut < 0.14))
    checks.append(("bill target below the fixed charge is impossible", L.heater_hours_for_bill(10.0) is None))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
